In [0]:
from pyspark.sql.functions import col, lit, current_timestamp

pipeline_name = "marketing_campaign_lakehouse_pipeline"
layer_name = "bronze"
table_name = "bronze_marketing_campaigns"

bronze_df = spark.table(table_name)

# registros inválidos por campos obrigatórios nulos
null_errors_df = bronze_df.filter(
    col("date").isNull() |
    col("campaign_id").isNull() |
    col("campaign_name").isNull()
).select(
    lit(pipeline_name).alias("pipeline_name"),
    lit(layer_name).alias("layer_name"),
    lit(table_name).alias("table_name"),
    lit("required_fields_not_null").alias("check_name"),
    lit("Required field is null").alias("error_reason"),
    col("campaign_id"),
    col("region"),
    col("date")
)

# registros inválidos por valores negativos
negative_errors_df = bronze_df.filter(
    (col("impressions") < 0) |
    (col("clicks") < 0) |
    (col("conversions") < 0) |
    (col("cost") < 0) |
    (col("revenue") < 0)
).select(
    lit(pipeline_name).alias("pipeline_name"),
    lit(layer_name).alias("layer_name"),
    lit(table_name).alias("table_name"),
    lit("numeric_fields_not_negative").alias("check_name"),
    lit("Negative numeric value found").alias("error_reason"),
    col("campaign_id"),
    col("region"),
    col("date")
)

all_errors_df = null_errors_df.unionByName(negative_errors_df) \
    .withColumn("run_timestamp", current_timestamp())

all_errors_df.write.format("delta") \
    .mode("append") \
    .insertInto("bronze_data_quality_errors")

# dados válidos: removem registros com erro
valid_bronze_df = bronze_df.filter(
    col("date").isNotNull() &
    col("campaign_id").isNotNull() &
    col("campaign_name").isNotNull() &
    (col("impressions") >= 0) &
    (col("clicks") >= 0) &
    (col("conversions") >= 0) &
    (col("cost") >= 0) &
    (col("revenue") >= 0)
)

valid_bronze_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_marketing_campaigns_valid")

error_count = all_errors_df.count()

print(f"Data quality completed. Invalid rows found: {error_count}")
print("Valid rows saved to bronze_marketing_campaigns_valid.")
print("Invalid rows saved to bronze_data_quality_errors.")